<a href="https://colab.research.google.com/github/Mia-NK/Mia_INFO4670_Fall2026/blob/main/INFO4670_Assignment_2_%5BDATA_CLEANING%2C_PREPROCESSING%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**INFO4670 ASSIGNMENT 2 - CLEANING AND EXTRACTING INFO**

In [658]:
import pandas as pd
import numpy as np

from scipy.stats import zscore
from sklearn.impute import SimpleImputer

In [659]:
studentsdf_og = pd.read_csv('student_records.csv')
enrolldf_og = pd.read_csv('course_enrollments.csv')
activitydf_og = pd.read_csv('weekly_activity.csv')

display(studentsdf_og.head(2),
        enrolldf_og.head(2),
        activitydf_og.head(2))

,student_id,major,age,housing,commute_miles,work_hours_per_week,advisor_meetings,credits_attempted,final_gpa,study_hours_reported,enrollment_date,dropped_out
0,NU-101508,Data Science,27,Off-Campus,7.6,15,1,81,1.86,NaN,2023-09-13,1
1,NU-102438,Learning Technologies,25,With Family,14.3,35,1,100,2.06,9.5,2023-10-19,0


,sid,course,term,grade
0,108933,INFO 2200,Spring 2025,D
1,109441,INFO 1100,Spring 2025,A


,student_id,week,lms_clicks,minutes_active
0,NU-101028,1,420,260
1,NU-101028,2,388,255


In [660]:
studentsdf_clean = studentsdf_og.copy()
enrolldf_clean = enrolldf_og.copy()
activitydf_clean = activitydf_og.copy()

display('STUDENTS:', studentsdf_clean.head(),
        'ENROLLMENTS:', enrolldf_clean.head(),
        'WEEKLY ACTIVITY:', activitydf_clean.head())

'STUDENTS:'

,student_id,major,age,housing,commute_miles,work_hours_per_week,advisor_meetings,credits_attempted,final_gpa,study_hours_reported,enrollment_date,dropped_out
0,NU-101508,Data Science,27,Off-Campus,7.6,15,1,81,1.86,NaN,2023-09-13,1
1,NU-102438,Learning Technologies,25,With Family,14.3,35,1,100,2.06,9.5,2023-10-19,0
2,NU-102385,Learning Technologies,26,With Family,3.4,18,9,77,3.41,19.1,11/16/2022,0
3,NU-100767,Cyber Security,22,On-Campus,23.7,40,1,65,1.73,8.1,2024-05-03,0
4,NU-106054,Information Technology,21,Off-Campus,2.3,20,4,59,2.37,6.6,10/24/2022,0


'ENROLLMENTS:'

,sid,course,term,grade
0,108933,INFO 2200,Spring 2025,D
1,109441,INFO 1100,Spring 2025,A
2,102076,DSCI 3210,Fall 2024,C
3,106381,CSEC 3220,Spring 2025,D
4,106185,CSEC 3220,Fall 2024,B


'WEEKLY ACTIVITY:'

,student_id,week,lms_clicks,minutes_active
0,NU-101028,1,420,260
1,NU-101028,2,388,255
2,NU-101028,3,544,340
3,NU-101028,4,498,311
4,NU-101028,5,319,266


# **1.1 Missing values**
Find every column with blanks. Choose one of Han's six methods for study_hours_reported and justify it (why that method, given the column's shape). Report the sample size before and after.

In [661]:
print('BLANKS\n\nSTUDENTS:', studentsdf_clean.isnull().sum(),
      '\n\nENROLLMENTS:', enrolldf_clean.isnull().sum(),
      '\n\nWEEKLY ACTIVITY:', activitydf_clean.isnull().sum())

BLANKS

STUDENTS: student_id                0
major                     0
age                       0
housing                   0
commute_miles             0
work_hours_per_week       0
advisor_meetings          0
credits_attempted         0
final_gpa                 0
study_hours_reported    255
enrollment_date           0
dropped_out               0
dtype: int64 

ENROLLMENTS: sid       0
course    0
term      0
grade     0
dtype: int64 

WEEKLY ACTIVITY: student_id        0
week              0
lms_clicks        0
minutes_active    0
dtype: int64


Blanks only discovered in the students df, and solely in the study_hours_reported col

In [662]:
mean_imp = SimpleImputer(missing_values=np.nan, strategy='mean')
#using the sklearn imputer to quickly fill in NaN values

studentsdf_clean[['study_hours_reported']] = mean_imp.fit_transform(
                         studentsdf_clean[['study_hours_reported']])
#fitting the imputer to the only col that had NaNs
studentsdf_clean[['study_hours_reported']] = studentsdf_clean[
                         ['study_hours_reported']].round(1)
#rounding the data to match the OG df (otherwise there will be lots of decimals)

display(studentsdf_og['study_hours_reported'].head(10),
    studentsdf_clean['study_hours_reported'].head(10))

,study_hours_reported
0,NaN
1,9.5
2,19.1
3,8.1
4,6.6
5,7.7
6,2.9
7,15.1
8,13.4
9,10.2


,study_hours_reported
0,10.5
1,9.5
2,19.1
3,8.1
4,6.6
5,7.7
6,2.9
7,15.1
8,13.4
9,10.2


Used mean imputation because the data are numerical and continuous; mode would really only be suited for categorical data, and median does not capture the averages in the data. By using the mean, we can get a good approximation of what the value likely would have been outside of ML intervention

# 1.2 Inconsistent categories
Standardize the housing column to its true groups. Show the value counts before and after.

In [663]:
studentsdf_clean['housing'].unique()

array(['Off-Campus', 'With Family', 'On-Campus', 'on-campus',
       ' On-Campus', 'off campus', 'Off-campus', 'with family'],
      dtype=object)

In [664]:
studentsdf_clean['housing'] = studentsdf_clean['housing'].str.lower()
studentsdf_clean['housing'] = studentsdf_clean['housing'].str.replace(' ', '-')
studentsdf_clean['housing'].unique() #check for any remaining strings that need
# more hands-on cleaning

array(['off-campus', 'with-family', 'on-campus', '-on-campus'],
      dtype=object)

In [665]:
studentsdf_clean.loc[studentsdf_clean['housing'] == '-on-campus',
                     'housing'] = 'on-campus'
studentsdf_clean['housing'].unique()

array(['off-campus', 'with-family', 'on-campus'], dtype=object)

In [666]:
print('OG data:', studentsdf_og['housing'].value_counts(),
'\n\nCLEAN data:', studentsdf_clean['housing'].value_counts())

OG data: housing
Off-Campus     755
On-Campus      480
With Family    420
off campus     113
Off-campus      77
with family     72
on-campus       69
 On-Campus      41
Name: count, dtype: int64 

CLEAN data: housing
off-campus     945
on-campus      590
with-family    492
Name: count, dtype: int64


# 1.3 Errors vs. extremes
Identify impossible values (check age and work hours). For each, decide fix or remove and say why — and explain how an impossible value differs from one that is extreme but valid.

In [667]:
display(studentsdf_clean['age'].unique(),
        studentsdf_clean['work_hours_per_week'].unique())

array([ 27,  25,  26,  22,  21,  24,  28,  20,  23,  29,   0,  32,  30,
        31,  34,  19,  33,  39,  38,   1,  36,  35,  37,  18, 199, -22,
       220,   3])

array([ 15,  35,  18,  40,  20,  25,  28,   0,  10,  12,   5,  30,  -8,
       -10,  -5, -12])

In [668]:
print((studentsdf_clean['age'] < 15).value_counts(),
      (studentsdf_clean['age'] > 200).value_counts())
#seeing how many of the data are impossible vals

age
False    2023
True        4
Name: count, dtype: int64 age
False    2026
True        1
Name: count, dtype: int64


These values only make up a marginal proportion (0.25%) of the data, so we can safely omit them; if desired, fixing *could* be an option in the case of entries like '199' or '220' where you can presume a button was accidentally hit twice as well, but negative values must be omitted

In [669]:
studentsdf_clean = studentsdf_clean[
    ~((studentsdf_clean['age'] < 15 ) |
     (studentsdf_clean['age'] > 100 ))]
#only retain data that lie between these values

studentsdf_clean = studentsdf_clean[
    ~((studentsdf_clean['work_hours_per_week'] < 0 ))]
#only retain data greater than this val

display(studentsdf_clean['age'].unique(),
        studentsdf_clean['work_hours_per_week'].unique())

array([27, 25, 26, 22, 21, 24, 28, 20, 23, 29, 32, 30, 31, 34, 19, 33, 39,
       38, 36, 35, 37, 18])

array([15, 35, 18, 40, 20, 25, 28,  0, 10, 12,  5, 30])

An impossible value is distinguished from a valid outlier based on whether it is possible [even if highly unlikely] to occur in reality. For instance, university students being as young as 15 is highly unlikely, but has still has occurred enough times in real life to be feasibly legitimate. Working negative hours, however, is impossible -- therefore we can understand these entries to likewise be impossible.

# 1.4 Detect and remove duplicate student records
In one sentence, explain why you do not de-duplicate course_enrollments or weekly_activity (think about what one row means in each file).

In [670]:
print('Student DF dupes pre-cleaning:', studentsdf_clean.duplicated(subset='student_id').sum())

studentsdf_clean.drop_duplicates(subset='student_id', inplace=True)

print('Student DF dupes post-cleaning:', studentsdf_clean.duplicated(subset='student_id').sum())

Student DF dupes pre-cleaning: 27
Student DF dupes post-cleaning: 0


weekly_activity is a panel dataset, i.e., it tracks individual data points over time, therefore the students/IDs repeat as the 'week' increments and changes in their activity are measured. In a regular 16-week course, you would see one student record 16 times, once for every week. Course_enrollments operates similarly, wherein all student enrollments in NorthGate are documented. This means that the same students/IDs will be shown multiple times if they enroll for >1 class. TLDR, the functionality and basic structure of these datasets is contingent on their repetition of records, and removing duplicates would gut them of the vast majority of their data and thereby information.

# 2.1 Join the files
Standardize the key so the files can be joined, then join student_records with course_enrollments and a per-student summary of weekly_activity (e.g., total minutes_active or total logins) into one table with one row per student.

In [671]:
analysis = studentsdf_clean.copy()

In [672]:
course_agg = enrolldf_clean.groupby('sid', as_index=False).agg({
    'course': 'count'})
activity_agg = activitydf_clean.groupby('student_id', as_index=False).agg({
    'minutes_active': 'sum'})

display(course_agg.head(),
        activity_agg.head())

,sid,course
0,100007,5
1,100015,5
2,100020,3
3,100021,3
4,100022,5


,student_id,minutes_active
0,NU-100007,2068
1,NU-100015,4586
2,NU-100020,1122
3,NU-100021,2042
4,NU-100022,3509


In [673]:
enrolldf_og[enrolldf_og['sid'] == 100015]
#double checking value counts to ensure they reflect the actual data

,sid,course,term,grade
561,100015,INFO 2200,Spring 2025,A
3236,100015,INFO 4300,Spring 2024,C
3833,100015,INFO 1100,Fall 2024,A
3859,100015,INFO 3600,Fall 2023,A
5564,100015,INFO 2600,Fall 2024,C


In [674]:
course_agg = course_agg.rename(columns={'sid': 'student_id',
                               'course': 'courses_enrolled'})

course_agg['student_id'] = 'NU-' + course_agg['student_id'].astype(str)
course_agg['student_id'] = course_agg['student_id'].astype(object)

analysis = pd.merge(analysis, activity_agg, how='left', on='student_id')
analysis = pd.merge(analysis, course_agg, how='left', on='student_id')
analysis.head()

,student_id,major,age,housing,commute_miles,work_hours_per_week,advisor_meetings,credits_attempted,final_gpa,study_hours_reported,enrollment_date,dropped_out,minutes_active,courses_enrolled
0,NU-101508,Data Science,27,off-campus,7.6,15,1,81,1.86,10.5,2023-09-13,1,2750,5
1,NU-102438,Learning Technologies,25,with-family,14.3,35,1,100,2.06,9.5,2023-10-19,0,2792,4
2,NU-102385,Learning Technologies,26,with-family,3.4,18,9,77,3.41,19.1,11/16/2022,0,5024,5
3,NU-100767,Cyber Security,22,on-campus,23.7,40,1,65,1.73,8.1,2024-05-03,0,2719,3
4,NU-106054,Information Technology,21,off-campus,2.3,20,4,59,2.37,6.6,10/24/2022,0,2916,4


# 2.2 Verify the join
Report how many rows matched, how many orphan IDs appeared, and confirm the final row count equals your cleaned roster.

In [675]:
enrolldf_og['sid'].dtype, analysis['student_id'].dtype

(dtype('int64'), dtype('O'))

In [676]:
match_ids = enrolldf_og['sid']
match_ids = match_ids.astype(str)
match_ids = 'NU-' + match_ids
match_ids = match_ids.astype(object)
#'sid' doesn't use same labeling scheme as the other dataframes' PKs, so to
# taking extra time to restructure the entire df I'm just taking and reformatting
# its IDs

print('No. matching IDs (enroll DF and student DF)',
      match_ids.isin(analysis['student_id']).sum(),
      '\nNo. records in enrolldf:', len(enrolldf_clean['sid']),
      '\n\nNo. matching IDs (activity DF and student DF)',
      activitydf_clean['student_id'].isin(analysis['student_id']).sum(),
      '\nNo. records in activitydf:', len(activitydf_clean['student_id']))

print('\n\nNo. records in students DF:', len(studentsdf_clean),
      '\nNo. records in analysis DF:', len(analysis))

No. matching IDs (enroll DF and student DF) 8002 
No. records in enrolldf: 8088 

No. matching IDs (activity DF and student DF) 31840 
No. records in activitydf: 32000


No. records in students DF: 1990 
No. records in analysis DF: 1990


Between the activity and enrollments dataframes and the analysis table, there are 246 total orhpnaed IDs.

The records in the students and analysis DFs matched completely, meaning no data were lost between them.

# 3.1 Clean date data
Parse enrollment_date so every date is recognized. Confirm no valid date was silently lost (count blanks before and after).

In [677]:
analysis['enrollment_date']

,enrollment_date
0,2023-09-13
1,2023-10-19
2,11/16/2022
3,2024-05-03
4,10/24/2022
...,...
1985,03/30/2024
1986,2023-09-23
1987,11/17/2022
1988,04-Apr-2024


In [678]:
pre_clean_dates = len(analysis['enrollment_date'])
analysis['enrollment_date'] = pd.to_datetime(analysis['enrollment_date'],
                              format='mixed', errors='coerce')

display(analysis['enrollment_date'])
print("\nOG NaN dates:", pre_clean_dates, "\nCLEAN NaN dates:",
      len(analysis['enrollment_date']))

,enrollment_date
0,2023-09-13
1,2023-10-19
2,2022-11-16
3,2024-05-03
4,2022-10-24
...,...
1985,2024-03-30
1986,2023-09-23
1987,2022-11-17
1988,2024-04-04



OG NaN dates: 1990 
CLEAN NaN dates: 1990


# 3.2 Add analysis columns
Add two columns: one normalized (z-score a numeric column) and one discretized (bin final_gpa into bands).

In [679]:
analysis['normalized_activity'] = zscore(analysis['minutes_active'])
#using scipy.stats to quickly create and populate z-score col using the
# minutes_active col

discretized_gpa = analysis.copy()
discretized_gpa['banded_gpa'] = pd.cut(analysis['final_gpa'], bins=[-0.01,1,2,3,4],
                         labels=['0-1', '1-2','2-3','3-4'])
analysis = pd.merge(analysis, discretized_gpa[['student_id', 'banded_gpa']],
                    how='left', on='student_id')

analysis.head()

,student_id,major,age,housing,commute_miles,work_hours_per_week,advisor_meetings,credits_attempted,final_gpa,study_hours_reported,enrollment_date,dropped_out,minutes_active,courses_enrolled,normalized_activity,banded_gpa
0,NU-101508,Data Science,27,off-campus,7.6,15,1,81,1.86,10.5,2023-09-13,1,2750,5,-0.501164,1-2
1,NU-102438,Learning Technologies,25,with-family,14.3,35,1,100,2.06,9.5,2023-10-19,0,2792,4,-0.459507,2-3
2,NU-102385,Learning Technologies,26,with-family,3.4,18,9,77,3.41,19.1,2022-11-16,0,5024,5,1.754256,3-4
3,NU-100767,Cyber Security,22,on-campus,23.7,40,1,65,1.73,8.1,2024-05-03,0,2719,3,-0.531911,1-2
4,NU-106054,Information Technology,21,off-campus,2.3,20,4,59,2.37,6.6,2022-10-24,0,2916,4,-0.336520,2-3


# 4.1 Export new table
Write your clean, integrated table to a new CSV (leave the raw files untouched).

In [680]:
analysis.to_csv('INFO4670 Assignment 2 - Complete Student Analysis', index=False)

# 4.2 Cleaning log
In a markdown cell, list each decision you made and a one-line justification for it.

**1.1**
Used mean imputation for NaN study_hours, mean used for numerical, continuous data

**1.2**
Streamlined string standardization: made all strings lowercase and hyphenated except for exception (leading whitespace) which was handled with direct reassignment.

**1.3**
Range of extreme values in age based on the data itself, only retained data within this range. Any impossible values removed from work_hours_per_week.

**1.4**
Removed any duplicate IDs with drop_duplicates() from students dataset only, wherein student_id = PK.

**2.1**
Joined files by duplicating students, then adding the total courses enrolled per student, as well as the sum of their minutes active.

**2.2**
Checked for major discrepancy between datasets and found none beyond ~240 orphan IDs.


**3.1**
Standardized dates in analysis and found no blanks before or after.

**3.2**
Used stats to quickly compute new z-score column for activity, created 4 bins for students' final GPA for easier aggregation.

# 4.3 Final insights
Using the clean data, report the mean GPA and one relationship you find interesting, and note in one sentence how cleaning changed the picture compared with the raw data.

In [681]:
print('Mean OG GPA:', studentsdf_og['final_gpa'].mean().round(2),
      '\nMean cleaned GPA:', analysis['final_gpa'].mean().round(2))
#No difference, but we didn't perform any cleaning operations on this column,
# so this is unsurprising

r_OG_studyhours_GPA = (studentsdf_og['study_hours_reported'].corr(
    studentsdf_og['final_gpa'])).round(2)
r_clean_studyhours_GPA = (analysis['study_hours_reported'].corr(
    analysis['final_gpa'])).round(2)
print('\nOG correlation (R) between study hours and GPA:',
      r_OG_studyhours_GPA,'\nNEW correlation (R) between study hours and GPA:',
      r_clean_studyhours_GPA)

real_activity = (analysis['study_hours_reported'].corr(analysis['minutes_active'])).round(2)
print('\nCorrelation between self-reported study hours and actual tracked activity',
      real_activity)

Mean OG GPA: 2.31 
Mean cleaned GPA: 2.31

OG correlation (R) between study hours and GPA: 0.7 
NEW correlation (R) between study hours and GPA: 0.62

Correlation between self-reported study hours and actual tracked activity 0.75


The final mean GPA in the cleaned data was 2.31. This is identical to the original dataset, bt this is to be expected, since I didn't perform any cleaning operations/alterations directly on 'final_gpa'. I was interested in seeing if correlation between self reported study hours and GPA had changed, since I had used mean imputation to populate the former in the cleaned dataset, and I found that this correlation decreased slightly NaN values were imputed, though the correlation remained moderately positive. Lastly I wanted to see if the cleaned, self-reported study hours correlated with actual activity data, and these did actually correlate moderately positively.

Cleaning provided a more representative view of the data which thereby provided a more accurate understanding of correlations between its dimensions.